In [2]:
import torch
import librosa
import numpy as np
import glob
import os
from tqdm import tqdm

def patch_absolute_loudness(data_dir, sr=16000, n_fft=2048, hop_length=64, win_length=128):
    """
    既存の.ptファイルを読み込み、絶対スケール(ref=1.0)かつ
    窓長8ms(win_length=128)のLoudnessを再計算して上書き保存するスクリプト。
    """
    files = glob.glob(os.path.join(data_dir, "*.pt"))
    print(f"Found {len(files)} files. Starting patch...")

    # A特性のフィルターを事前に計算 (n_fft=2048で作るため高精度)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    a_weighting_db = librosa.A_weighting(freqs)
    a_weighting_linear = 10.0 ** (a_weighting_db / 10.0) # dBから線形パワー乗数へ

    for path in tqdm(files):
        data = torch.load(path, weights_only=False)
        audio = data['audio'].numpy()
        f0_len = data['f0'].shape[0]

        # 1. STFT (周波数解像度は2048、時間の窓は128で鋭く切り取る)
        stft = librosa.stft(
            y=audio,
            n_fft=n_fft,
            hop_length=hop_length,
            win_length=win_length,
            window='hann',
            center=True
        )
        power_spec = np.abs(stft) ** 2

        # 2. A特性フィルターを掛ける
        weighted_power = power_spec * a_weighting_linear[:, np.newaxis]

        # 3. 周波数方向の平均パワーを計算
        mean_power = np.mean(weighted_power, axis=0)

        # 4. 絶対スケールのdBに変換 (ref=1.0, ローカル正規化なし)
        # librosa.power_to_db のデフォルトの下限(amin)は 1e-10 (-100dB) です
        loudness_db = librosa.power_to_db(mean_power, ref=1.0, top_db=None)

        # 5. 長さを f0 に合わせる
        loudness_db = loudness_db[:f0_len]

        # テンソルに変換して上書き
        data['loudness'] = torch.from_numpy(loudness_db).unsqueeze(-1).float() # [T, 1]

        # 保存
        torch.save(data, path)

    print("Patch completed successfully!")

# 実行例（パスはご自身の環境に合わせて変更してください）
patch_absolute_loudness("/content/drive/MyDrive/hioki_lab/data/URMP_solo_inst/processed_data")

Found 968 files. Starting patch...


/usr/local/lib/python3.12/dist-packages/librosa/core/convert.py:1869: RuntimeWarning: divide by zero encountered in log10
  + 2 * np.log10(f_sq)
100%|██████████| 968/968 [03:25<00:00,  4.70it/s]

Patch completed successfully!


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
